# YOLOv8 com filtros e operadores de borda

Compare 11 variantes: RGB, cinza, dois Gaussianos, mediana, bilateral, dois Canny, Sobel, Laplaciano e RGB+Sobel. Os hiperparâmetros são fixos e os aumentos artificiais estão desativados para isolar o processamento.

Ative uma GPU. Envie o **novo colab_bundle.zip**, contendo os scripts desta comparação. O dataset original fica intacto; treino e validação recebem a mesma transformação. O teste permanece reservado. Uma variante por vez é gerada no disco temporário do Colab e removida após salvar o resultado.

Os resultados anteriores de hiperparâmetros não são sobrescritos. Não execute este notebook em duas sessões simultaneamente.


In [ ]:
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
assert "colab_bundle.zip" in uploaded, "Selecione colab_bundle.zip"
project = Path("/content/what-is-the-book-2")
project.mkdir(exist_ok=True)
with zipfile.ZipFile("colab_bundle.zip") as archive:
    assert all((project / name).resolve().is_relative_to(project) for name in archive.namelist())
    archive.extractall(project)
del uploaded
%cd /content/what-is-the-book-2

In [ ]:
import os

os.environ["MPLBACKEND"] = "Agg"
import subprocess

subprocess.run(["pip", "install", "uv"], check=True)
subprocess.run(
    ["uv", "sync", "--locked", "--python", "3.13", "--extra", "training", "--no-dev"], check=True
)
subprocess.run(
    [
        "uv",
        "run",
        "--no-sync",
        "python",
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda); "
        "assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/what-is-the-book-2/runs/preprocessing")
OUTPUT.mkdir(parents=True, exist_ok=True)
BASE = [
    "uv",
    "run",
    "--no-sync",
    "python",
    "-u",
    "-m",
    "src.detection.train_preprocessing",
    "--output",
    str(OUTPUT),
    "--cache",
    "/content/book_preprocessing_cache",
]

In [ ]:
import os
import subprocess
from collections import deque
from datetime import datetime


def executar(arguments, name):
    logs = OUTPUT / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = logs / f"{name}_{stamp}.log"
    print(f"Log completo: {log_path}", flush=True)
    tail = deque(maxlen=35)
    last_epoch = None
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            arguments,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "MPLBACKEND": "Agg", "PYTHONUNBUFFERED": "1"},
        )
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                tail.append(line.rstrip())
                parts = line.strip().split()
                token = parts[0] if parts else ""
                epoch = token.split("/")
                if "STATUS |" in line:
                    last_epoch = None
                    print(line.rstrip(), flush=True)
                elif len(epoch) == 2 and all(p.isdigit() for p in epoch):
                    if token != last_epoch:
                        last_epoch = token
                        print(f"Epoca {token}", flush=True)
                elif any(t in line for t in ("Error", "WARNING", "epochs completed")):
                    print(line.rstrip(), flush=True)
            status = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            print("Interrompido. Checkpoints persistidos foram preservados.")
            raise
    if status:
        print("\n".join(tail))
        raise RuntimeError(f"Falha no processo: {status}. Consulte {log_path}")
    print("Concluido.", flush=True)

## Inspecione as transformações

Cada painel mostra uma imagem de treino de uma origem, com todas as variantes. Os mapas de borda descartam textura/cor; isso pode ajudar ou prejudicar a detecção. Não há resultados de treinamento ainda.


In [ ]:
from IPython.display import Image, display

executar(BASE + ["--preview"], "preview")
for image in sorted((OUTPUT / "previews").glob("*.png")):
    print(image.name)
    display(Image(filename=str(image), width=1000))

## Teste curto opcional

Ative RUN_SMOKE para conferir o ambiente com uma época e 5% do treino da variante RGB. Não confunda essas métricas com os experimentos completos. O teste curto também pode ser retomado; se já concluiu, será pulado.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    executar(BASE + ["--run", "--smoke", "--variant", "original"], "smoke")

## Treinar ou retomar automaticamente

A próxima célula executa as 11 variantes sequencialmente. Cada uma usa até 50 épocas; a grade completa pode levar horas. Resultados completos são pulados; last.pt incompleto é retomado; treino finalizado sem result.json tem sua validação gerada.

Se a sessão cair, confira se não há outro processo ativo, restaure upload, ambiente e Drive, e execute novamente. Não mude configurações ou dados dentro de uma grade em andamento. O log integral fica no Drive, mas só resumos por época aparecem aqui.


In [ ]:
executar(BASE + ["--run"], "preprocessing")

In [ ]:
import json

ranking_path = OUTPUT / "grid/ranking.json"
if ranking_path.exists():
    ranking = json.loads(ranking_path.read_text())
    for position, result in enumerate(ranking, 1):
        name = result["request"]["variant"]["name"]
        metrics = result["metrics"]
        print(
            f"{position:2d}. {name:20s} "
            f"mAP50-95={metrics['metrics/mAP50-95(B)']:.4f} "
            f"mAP50={metrics['metrics/mAP50(B)']:.4f}"
        )
else:
    print("Nenhum resultado concluido ainda.")

## Depois dos experimentos

Compare com a referência RGB desta grade, pois a grade anterior usava aumentos diferentes. Examine também precisão, recall e exemplos visuais. Na inferência e na futura avaliação do teste, aplique exatamente a mesma função `transform` e os parâmetros da variante escolhida antes de enviar a imagem ao modelo.

Detalhes: `docs/experimentos_preprocessamento.md`.
